In [ ]:
import sys
print(sys.version)

import transformers
import datasets
import pandas

print("transformers:",transformers.__version__)
print("datasets:",datasets.__version__)
print("pandas:",pandas.__version__)
print("환경설정 완료")

### 모델 저장 경로
- C:\Users\Admin\.cache\huggingface\hub

# Access token 사용

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv() # .env 파일을 읽어와 환경 변수로 등록

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HUGGINGFACE_API_KEY"],
)

# print(os.environ["HUGGINGFACE_API_KEY"])
# client
# output = client.automatic_speech_recognition("sample1.flac", model="openai/whisper-large-v3")

In [2]:
import fastapi; 
import transformers; 
import torch; 

print('FastAPI:', fastapi.__version__); 
print('Transformers:', transformers.__version__); 
print('Torch:', torch.__version__); 
print('CUDA:', torch.cuda.is_available())

FastAPI: 0.135.3
Transformers: 5.4.0
Torch: 2.11.0+cu126
CUDA: True


In [ ]:
# 번역 전용 모델 사용 예시
result = client.translation(
    "안녕하세요, 반갑습니다.",
    model="Helsinki-NLP/opus-mt-ko-en"
)

print(result.translation_text)

In [ ]:
# device = 0 if torch.cuda.is_available() else -1, torch.inference_mode()

# device

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "sentiment-analysis",
    # model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0
)

result = pipe("I really enjoyed this class.")
print(result)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, MarianTokenizer # 클래스 직접 임포트

model_name = "Helsinki-NLP/opus-mt-ko-en"

# AutoTokenizer 대신 MarianTokenizer 사용
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

inputs = tokenizer("이 영화 정말 별로야.", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs)

# outputs는 2차원 텐서이므로 [0]으로 첫 번째 결과 선택
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

mdl = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device) 

inputs = tok("I love this.", return_tensors="pt").to(device)
with torch.inference_mode():
    logits = mdl(**inputs).logits
    print(logits)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
mdl = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
ids = tok("Write a haiku about summer:", return_tensors="pt").to(mdl.device)
out = mdl.generate(**ids, max_new_tokens=64)
print(tok.decode(out[0], skip_special_tokens=True))